# RIS / 03 — SOFR futures, 4th quarterly contract (`SR3C3`): cross-confirmation H1 ↔ H2

Protocol: [`RIS 001`](../../../discussions/Rate_Intraday_Seasonality/001_claude_ris01_preregistration.md), cross-confirmation as [`BDS 001`](../../../discussions/Bond_Intraday_Seasonality/001_claude_bds01_preregistration.md) §6. Reads the frozen candidates of `01/` (H1) and `02/` (H2) unchanged (SHA-256 checked against each `summary.json`).

- **Forward:** H1 candidates traded every H2 day (2024–2025).
- **Backward:** H2 candidates traded every H1 day (2022–2023).

One trade per day at the quoted bid/ask of the future plus 0.70 bp commission.

| Gate | Condition |
|---|---|
| K1 | mean after the tick > 0, Newey–West one-sided p, BH across the window's candidates, q ≤ 0.05 |
| K2 | net: block-bootstrap lower bound > 0 at blocks 10 and 20 |
| K3 | mean net at 2× commission > 0 |
| K4 | yearly mean net > 0 in ≥ 60% of counted years |

In [ ]:
SYMBOL = "SR3C3"

import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("RIS_SYNTHETIC") == "1"
if SYNTHETIC:                                   # never touch the real market-data folder in a smoke run
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("RIS_SYNTH_DIR") or tempfile.mkdtemp(prefix="ris_")
    os.environ["RIS_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F
from equity_trading import bond_seasonality as B

cfg = B.BDSConfig()
if SYNTHETIC:
    cfg = replace(cfg, n_placebo=3, n_boot=2_000)
INFO = B.info(SYMBOL)
SHORT = INFO["short"]
POOLED_T = F.bonferroni_t(len(F.intervals(cfg.fx())) * len(B.CLOCKS) * len(B.RATE_SYMBOLS))
HOURS = lambda clock, step: ([60 * h for h in range(0, 25, step)], [F.slot_label(clock, 60 * h).split()[1] for h in range(0, 25, step)])
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120)
SUF = "_synthetic" if SYNTHETIC else ""
OUT = Path.cwd() / ("03" + SUF); OUT.mkdir(exist_ok=True)
frozen = {}
for w, nn in (("H1", "01"), ("H2", "02")):
    c, s = B.load_frozen(Path.cwd() / (nn + SUF))
    frozen[w] = c
    print(f"{w}: {len(c)} frozen candidates (sha256 {s['candidates_sha256'][:12]}...), D0 = {s['D0']}")
PLAN = {"forward": ("H1", "H2"), "backward": ("H2", "H1")}

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"])
    for r, (px, drift, seed) in {"ZT": (105.0, 3.0, 21), "ZF": (115.0, 0.0, 22), "SR3C3": (96.0, 0.0, 23)}.items():
        if not (base / "Databento" / "bbo-1m" / r).exists():
            B.write_synthetic_rate(base, r, dt.date(2015, 12, 28), dt.date(2025, 12, 31), seed=seed, drift_bps=drift, price=px)

## 1. Found in both halves?

H1 and H2 candidates that overlap in UTC time with the same direction: found independently in each half.

In [ ]:
both = [{"H1": a["id"], "H2": b["id"], "H1 trade": f"{a['start_time']}-{a['end_time'].split()[1]}",
          "H2 trade": f"{b['start_time']}-{b['end_time'].split()[1]}", "direction": "long" if a["direction"] > 0 else "short"}
         for _, a in frozen["H1"].iterrows() for _, b in frozen["H2"].iterrows()
         if int(a["direction"]) == int(b["direction"]) and B.overlaps(a, b)]
both = pd.DataFrame(both)
display(both if len(both) else "no structure found in both halves"); both.to_csv(OUT / "03_found_in_both.csv", index=False)

## 2. Gates K1–K4

In [ ]:
data = {w: B.load(SYMBOL, w, cfg) for w in ("H1", "H2")}
results, trades = [], {}
for direction, (src, dst) in PLAN.items():
    cand = frozen[src]
    if cand.empty:
        print(f"{direction}: no {src} candidates"); continue
    panels = data[dst].panels
    tr = {r["id"]: B.trades(panels[r["clock"]], int(r["start"]), int(r["end"]), int(r["direction"]), cfg) for _, r in cand.iterrows()}
    trades.update(tr)
    res = cand[["id", "clock", "start_time", "end_time", "direction"]].assign(test=direction, on=dst, disc_mean_bps=cand["mean_bps"].abs()).merge(
        B.confirm(tr, panels, cfg), on="id")
    res["verdict"] = np.where(res["verdict"] == "CONFIRMED", f"CONFIRMED_{direction.upper()}", "NOT_CONFIRMED")
    results.append(res)
res = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
if len(res):
    display(res.drop(columns=["boot_net_lower", "gross_pips"]).round(3))
    print(res["verdict"].value_counts().to_string())
    res.to_csv(OUT / "03_cross_confirmation.csv", index=False)

## 3. By year and cumulative net

Yearly mean net (bps) on the confirmation window, and the cumulative net of one unit traded every day.

In [ ]:
if len(res):
    yearly = pd.DataFrame({cid: tr.groupby("year")["net_bps"].mean() for cid, tr in trades.items()}).T
    display(yearly.round(2)); yearly.to_csv(OUT / "03_by_year_net.csv")
    n = len(res)
    fig, axes = plt.subplots(int(np.ceil(n / 2)), 2, figsize=(16, 3 * np.ceil(n / 2)), squeeze=False)
    for ax, (_, r) in zip(axes.flat, res.iterrows()):
        tr = trades[r["id"]]
        ax.plot(pd.to_datetime(tr["date"]), tr["net_bps"].cumsum(), lw=0.9, color="C2" if r["verdict"].startswith("CONFIRMED") else "C3", label="futures cost")
        ax.plot(pd.to_datetime(tr["date"]), tr["cfd_net_bps"].cumsum(), lw=0.7, color="grey", label="net at quotes")
        ax.axhline(0, color="black", lw=0.5); ax.legend(fontsize=7)
        ax.set_title(f"{r['id']} on {r['on']} ({r['start_time']}-{r['end_time'].split()[1]}): {r['verdict']}", fontsize=9)
    for ax in axes.flat[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "03_cumulative.png", dpi=110); plt.show()

## 4. Discovery vs confirmation

If the discovery means were mostly selection noise, the confirmation means collapse towards zero.

In [ ]:
if len(res):
    fig, ax = plt.subplots(figsize=(6, 5))
    for d, mk in (("forward", "o"), ("backward", "s")):
        x = res[res["test"] == d]
        ax.scatter(x["disc_mean_bps"], x["gross_bps"], marker=mk, label=d,
                   c=np.where(x["verdict"].str.startswith("CONFIRMED"), "C2", "C3"))
    lim = max(1.0, float(res["disc_mean_bps"].max()))
    ax.plot([0, lim], [0, lim], "k--", lw=0.6); ax.axhline(0, color="black", lw=0.5); ax.legend()
    ax.set_xlabel("discovery mean in trade direction (bps, mid)"); ax.set_ylabel("confirmation mean after one tick (bps)")
    plt.tight_layout(); plt.savefig(OUT / "03_shrinkage.png", dpi=110); plt.show()

In [ ]:
summary = {"study": "RIS01-cross", "symbol": SYMBOL, "bds_version": B.BDS_VERSION, "synthetic": SYNTHETIC,
           "frozen": {w: int(len(c)) for w, c in frozen.items()}, "found_in_both": both.to_dict(orient="records"),
           "confirmed": res.loc[res["verdict"].str.startswith("CONFIRMED"), ["id", "verdict"]].to_dict(orient="records") if len(res) else [],
           "results": res.drop(columns=["boot_net_lower"]).to_dict(orient="records") if len(res) else []}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: summary[k] for k in ("symbol", "frozen", "confirmed")}, indent=2))